# 01~06 코드 빈칸 채우기 테스트

기존 실전문제와 오답노트에서 반복해서 헷갈린 코드를 다시 구성한 테스트입니다.

- `____`를 올바른 코드로 바꾸세요.
- 각 세트는 독립적으로 실행할 수 있습니다.
- 정답은 `answers/01_06_fill_in_answer.ipynb`에 있습니다.
- 먼저 정답을 보지 않고 작성한 뒤, 결과의 shape와 최종값까지 확인하세요.


## Set 1 — 문자열 추출과 Random Forest

핵심: 문자열 열 결합, 정규식 추출, OR 검색, 1차원 y, 변수 중요도


In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

df = pd.read_csv('../../dataset/TV.csv')
df['spec_text'] = df[['Picture_quality', 'Speaker', 'Frequency']].agg(' '.join, axis=1)

df['refresh_rate'] = df['spec_text'].str.____(
    r'(?P<refresh_rate>\d{2,3})\s*Hz',
    expand=____
)

df['high_quality'] = df['Picture_quality'].str.____(
    r'4K____8K',
    na=False
).astype(int)

df['review_ratio'] = df['Reviews'] / df['Ratings']
df['price_ratio'] = df['current_price'] / df['MRP']
model_df = df.dropna(subset=['review_ratio', 'price_ratio']).copy()

cols_X = ['review_ratio', 'MRP', 'price_ratio', 'high_quality']
X = model_df[cols_X]
y = model_df____'Stars'____

model = RandomForestRegressor(random_state=123)
model.____(X, y)

importance = pd.Series(model.____, index=cols_X)
display(importance.____())


## Set 2 — 정확한 범주 검사와 상관행렬

핵심: `isin`, 행 단위 all, 몫, 자기상관 제외, Series y


In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

df = pd.read_csv('../../dataset/galaxy_users.csv')
service_cols = [
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies'
]

valid = df[service_cols].____(['Yes', 'No']).____(axis=1)
base = df.loc[____valid].copy()
base[service_cols] = base[service_cols].____({'Yes': 1, 'No': 0}).astype(int)
base['service_count'] = base[service_cols].____(axis=1)
base['used_month'] = base['TotalCharges'] ____ base['MonthlyCharges']

corr_abs = base[['tenure', 'MonthlyCharges', 'used_month']].corr().____()
np.fill_diagonal(corr_abs.____, 0)
display(corr_abs.max().____())

X = base[['tenure', 'MonthlyCharges', 'TotalCharges', 'service_count']]
y = base____'Stars'____ if 'Stars' in base.columns else base['Churn']


## Set 3 — 이상치, OHE, k-NN RMSE

핵심: 상위 이상치, 명시적 더미화, train 기반 정규화, RMSE 최소화


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error

df = pd.read_csv('../../dataset/mobiles.csv')

upper = df['sales'].____() + 2 * df['sales'].____()
focus = df.loc[df['sales'] ____ upper].copy()

X = df.drop(columns='sales').copy()
y = df['sales']
X = pd.____(X, columns=____'screen_size'____, drop_first=False)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=123
)

scaler = MinMaxScaler()
X_train_scaled = scaler.____(X_train)
X_test_scaled = scaler.____(X_test)

rmse_by_k = {}
for k in [3, 5, 7, 9, 11]:
    model = KNeighborsRegressor(n_neighbors=k)
    model.fit(X_train_scaled, y_train)
    pred = model.predict(X_test_scaled)
    rmse_by_k[k] = mean_squared_error(y_test, pred) ____ 0.5

rmse = pd.Series(rmse_by_k)
display(rmse.____())


## Set 4 — 고객 단위 Named Aggregation과 군집평가

핵심: `first`, `nunique`, `sum`, 식별자 제외, OHE, Silhouette score


In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

df = pd.read_csv('../../dataset/sales_pos.csv')

df_user = df.groupby('user').agg(
    gender=('gender', '____'),
    age_group=('age_group', '____'),
    job=('job', '____'),
    city=('city', '____'),
    marital=('marital', '____'),
    prod_count=('prod', '____'),
    total_purchase=('purchase', '____')
)

df_user['gender'] = df_user['gender'].____({'M': 1, 'F': 0})
df_user['age_group'] = pd.to_numeric(
    df_user['age_group'].str.____(r'(\d+)', expand=False)
)
X = pd.get_dummies(df_user, columns=____'job', 'city'____)

scaler = MinMaxScaler()
X_scaled = scaler.____(X)

model = KMeans(n_clusters=7, random_state=123, n_init=10)
labels = model.____(X_scaled)
score = silhouette_score(____, ____)
display(round(score, 2))


## Set 5 — 그룹별 상관계수, 군집 라벨, 조건 분할

핵심: GroupBy apply, StandardScaler, 최적 k 라벨, modulo 분할, RMSE


In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, mean_squared_error
from sklearn.tree import DecisionTreeRegressor

df = pd.read_csv('../../dataset/card_cust.csv')
base = df.copy()
base['MINIMUM_PAYMENTS'] = base['MINIMUM_PAYMENTS'].____(
    base['MINIMUM_PAYMENTS'].____()
)

corr_by_tenure = base.groupby('TENURE').____(
    lambda group: group['BALANCE'].____(group['CREDIT_LIMIT'])
)

X = base.drop(columns='CUST_ID').copy()
X_scaled = StandardScaler().____(X)

scores = {}
labels_by_k = {}
for k in range(2, 6):
    labels = KMeans(n_clusters=k, random_state=1234).____(X_scaled)
    scores[k] = silhouette_score(X_scaled, labels)
    labels_by_k[k] = labels

best_k = pd.Series(scores).____()
X['cluster'] = labels_by_k[____]

train = base.loc[base['CUST_ID'] % 4 ____ 0].copy()
test = base.loc[base['CUST_ID'] % 4 ____ 0].copy()
model = DecisionTreeRegressor(random_state=1234)


## Set 6 — 문자형 결측치, 마지막 기준범주, Odds Ratio, Accuracy

핵심: `select_dtypes`, `dropna(subset)`, 마지막 더미열 제거, `exp(coef)`, 혼동행렬


In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

df = pd.read_csv('../../dataset/edu_enrollees.csv')
base = df.drop(columns=['city', 'company_size', 'company_type']).copy()

object_cols = base.____(include='object').columns
base = base.____(subset=object_cols).copy()
base = base.loc[____base['experience'].isin(['>20', '<1'])].copy()
base['experience'] = base['experience'].astype(____)

dummy = pd.get_dummies(base['gender'], prefix='gender')
dummy = dummy.iloc[:, ____].copy()  # 사전순 마지막 열 제외

target_rate = base.groupby('relevant_experience')['target'].____()

categorical_cols = [
    'gender', 'relevant_experience', 'enrolled_university',
    'education_level', 'major_discipline', 'last_new_job'
]
numeric_cols = ['city_development_index', 'experience', 'training_hours']
job2 = pd.get_dummies(
    base[numeric_cols + categorical_cols + ['target', 'Xgrp']],
    columns=categorical_cols,
    drop_first=____
)
X = job2.drop(columns=['target', 'Xgrp'])
y = job2['target']
model = LogisticRegression(C=100000, max_iter=1000, solver='liblinear', random_state=123)
model.fit(X, y)

odds_ratio = pd.Series(np.____(model.coef_[0]), index=X.columns)
answer = np.____(odds_ratio.max() * 100) / 100

train = job2.loc[job2['Xgrp'] ____ 'train'].copy()
test = job2.loc[job2['Xgrp'] ____ 'test'].copy()
knn = KNeighborsClassifier(n_neighbors=5, metric='____')
